# Практическое занятие 7. Создание базы данных SQL и работа с данными

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/p07.ipynb)

**Цели занятия:**

- Создавать базу данных SQLite и таблицы с типами столбцов и первичным ключом
- Добавлять данные безопасно — через параметры `?` и `executemany`
- Писать запросы SELECT с WHERE, ORDER BY, GROUP BY, HAVING, JOIN, а также UPDATE и DELETE
- Связывать SQL и pandas: `pd.read_sql_query` и `DataFrame.to_sql`

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/OWNER/REPO.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Реляционная база данных

**Реляционная БД** хранит данные в **таблицах** со строгой схемой: у каждого столбца есть имя и тип, у каждой строки — уникальный **первичный ключ** (PRIMARY KEY). Таблицы связываются друг с другом через ключи, а запросы пишутся на языке **SQL**.

**SQLite** — встроенная СУБД: вся база — это один файл (например, `university.db`), сервер не нужен. Модуль `sqlite3` входит в стандартную библиотеку Python. Для больших промышленных систем используют PostgreSQL, MySQL, а в мире Big Data — SQL-движки поверх кластеров (Hive, Spark SQL), но язык запросов почти тот же.

| Тип SQLite | Что хранит | Пример |
|---|---|---|
| `INTEGER` | целые числа | курс, год |
| `REAL` | дробные числа | GPA |
| `TEXT` | строки | ФИО, город |
| `BLOB` | двоичные данные | файл, изображение |
| `NULL` | отсутствие значения | пропуск |

## Создание таблицы и вставка

```python
import sqlite3

conn = sqlite3.connect("university.db")     # файл создаётся, если его нет
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id TEXT PRIMARY KEY,
    full_name  TEXT NOT NULL,
    faculty    TEXT,
    course     INTEGER,
    gpa        REAL
)""")
cur.execute("INSERT INTO students VALUES (?, ?, ?, ?, ?)",
            ("S0001", "Saidov Otabek", "Kompyuter lingvistikasi", 2, 4.13))
rows = [("S0002", "Tursunova Munisa", "Tarjimonlik", 1, 4.22),
        ("S0003", "Nazarova Sevara", "Xalqaro jurnalistika", 3, 4.6)]
cur.executemany("INSERT INTO students VALUES (?, ?, ?, ?, ?)", rows)
conn.commit()                               # без commit изменения не сохранятся!
```

> **Внимание: Только параметры «?»**  
> Никогда не вставляйте значения в SQL через f-строки: `f"... WHERE city = '{city}'"`. Если в значении окажется кавычка (`Farg'ona`!), запрос сломается, а злоумышленник сможет выполнить свой SQL (**SQL-инъекция**). Параметры `?` экранируют значения автоматически.

Повторная вставка того же `student_id` вызовет `sqlite3.IntegrityError` — первичный ключ гарантирует уникальность.

## Запросы SELECT

```sql
SELECT full_name, gpa                -- какие столбцы
FROM students                        -- из какой таблицы
WHERE faculty = 'Filologiya'         -- фильтр строк
ORDER BY gpa DESC                    -- сортировка
LIMIT 5;                             -- первые 5

SELECT city, COUNT(*) AS n, AVG(gpa) AS avg_gpa
FROM students
GROUP BY city                        -- группировка
HAVING COUNT(*) >= 15;               -- фильтр групп (после GROUP BY)
```

В Python: `cur.execute(sql, params).fetchall()` возвращает список кортежей, `fetchone()` — один кортеж.

| SQL | pandas |
|---|---|
| `WHERE gpa > 4` | `df[df["gpa"] > 4]` |
| `ORDER BY gpa DESC LIMIT 5` | `df.nlargest(5, "gpa")` |
| `GROUP BY city` + `AVG(gpa)` | `df.groupby("city")["gpa"].mean()` |
| `HAVING COUNT(*) >= 15` | фильтр после `groupby` |
| `JOIN ... ON` | `pd.merge(a, b, on=...)` |

## JOIN — соединение таблиц

```sql
SELECT s.order_id, s.qty, b.category
FROM sales AS s
JOIN books AS b ON s.book = b.book;            -- только совпавшие строки

SELECT b.book FROM books AS b
LEFT JOIN sales AS s ON s.book = b.book
WHERE s.order_id IS NULL;                      -- книги без продаж
```

## Изменение и удаление

```python
cur.execute("UPDATE students SET gpa = gpa + 0.1 WHERE attendance >= ?", (95,))
print(cur.rowcount)                            # сколько строк изменено
cur.execute("DELETE FROM students WHERE course = ?", (4,))
conn.commit()
```

## SQL и pandas вместе

```python
import pandas as pd

df = pd.read_csv("data/students.csv")
df.to_sql("students", conn, if_exists="replace", index=False)     # DataFrame → таблица
top = pd.read_sql_query("SELECT * FROM students WHERE gpa >= ?", conn, params=(4.5,))
conn.close()
```

> **Заметка: Как устроен тренажёр**  
> Файлы, которые создаёт ваш код (например, `university.db`), живут в памяти браузера до перезагрузки страницы. Подготовительный код заданий каждый раз пересоздаёт базу, поэтому задания можно запускать повторно.

## Задание 1. Создание базы и таблицы

Схема таблицы `students`:

| Столбец | Тип | Ограничение |
|---|---|---|
| `student_id` | TEXT | PRIMARY KEY |
| `full_name` | TEXT | NOT NULL |
| `faculty` | TEXT | |
| `course` | INTEGER | |
| `city` | TEXT | |
| `gpa` | REAL | |

1. Подключитесь к базе `university.db` (`conn`) и создайте таблицу `students` по этой схеме.
2. Добавьте **три** студента запросом `INSERT` с параметрами `?` и сохраните изменения (`commit`).
3. Попробуйте ещё раз вставить студента с уже существующим `student_id`; перехватите `sqlite3.IntegrityError` и сохраните текст ошибки в `err`.
4. `n` — число строк в таблице (`SELECT COUNT(*)`). Закройте соединение.

<details><summary>💡 Подсказка</summary>

Пример вставки: `cur.execute("INSERT INTO students VALUES (?, ?, ?, ?, ?, ?)", ("S0001", "Saidov Otabek", "Kompyuter lingvistikasi", 2, "Andijon", 4.13))`. Перехват: `try: ... except sqlite3.IntegrityError as e: err = str(e)`.

</details>

In [ ]:
# Подготовительный код задания
import os
if os.path.exists("university.db"):
    os.remove("university.db")

In [ ]:
import sqlite3

conn = sqlite3.connect("university.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE students (
    student_id TEXT PRIMARY KEY
    -- допишите остальные столбцы
)""")

# добавьте трёх студентов через параметры ?

err = None
# попробуйте вставить дубликат student_id и перехватите sqlite3.IntegrityError

n = ...
print("Строк:", n, "| ошибка:", err)
conn.close()

## Задание 2. Массовая загрузка: executemany

В базе `university.db` уже есть **пустая** таблица `students` со столбцами
`student_id, full_name, gender, faculty, course, city, gpa, attendance, english_level`.

1. Прочитайте `data/students.csv` (pandas или модуль `csv`).
2. Подготовьте `rows` — список кортежей со значениями этих 9 столбцов **в том же порядке**.
3. Вставьте все строки **одним** вызовом `executemany` и выполните `commit`.
4. `n` — число строк в таблице после загрузки.

<details><summary>💡 Подсказка</summary>

`rows = list(df[cols].itertuples(index=False, name=None))`, затем `conn.executemany("INSERT INTO students VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", rows)` и `conn.commit()`.

</details>

In [ ]:
# Подготовительный код задания
import os, sqlite3
if os.path.exists("university.db"):
    os.remove("university.db")
_c = sqlite3.connect("university.db")
_c.execute("""CREATE TABLE students (
    student_id TEXT PRIMARY KEY, full_name TEXT NOT NULL, gender TEXT,
    faculty TEXT, course INTEGER, city TEXT, gpa REAL, attendance REAL, english_level TEXT)""")
_c.commit()
_c.close()
del _c

In [ ]:
import sqlite3
import pandas as pd

cols = ["student_id", "full_name", "gender", "faculty", "course", "city", "gpa", "attendance", "english_level"]
df = pd.read_csv("data/students.csv")

rows = ...

conn = sqlite3.connect("university.db")
# вставьте rows через executemany и сохраните изменения

n = conn.execute("SELECT COUNT(*) FROM students").fetchone()[0]
print("Загружено строк:", n)
conn.close()

## Задание 3. SELECT: WHERE, ORDER BY, LIMIT

Соединение `conn` с таблицей `students` (все 12 столбцов из `students.csv`) и курсор `cur` уже готовы. Напишите запросы SQL:

- `top5` — список кортежей `(full_name, gpa)`: пять лучших по GPA студентов факультета **Kompyuter lingvistikasi**, по убыванию GPA (название факультета передайте параметром `?`);
- `n_tash` — сколько студентов из города **Toshkent** имеют уровень английского `B2` или `C1` (`IN (...)`), одним числом;
- `by_course` — список кортежей `(course, средний gpa)` по всем курсам, средний GPA округлите до 2 знаков (`ROUND(AVG(gpa), 2)`), сортировка по курсу.

<details><summary>💡 Подсказка</summary>

`cur.execute("SELECT full_name, gpa FROM students WHERE faculty = ? ORDER BY gpa DESC LIMIT 5", ("Kompyuter lingvistikasi",)).fetchall()`. Одно число: `.fetchone()[0]`.

</details>

In [ ]:
# Подготовительный код задания
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
pd.read_csv("data/students.csv").to_sql("students", conn, index=False)
cur = conn.cursor()

In [ ]:
top5 = cur.execute("""
    SELECT ...
""").fetchall()

n_tash = ...

by_course = ...

print(top5)
print("Ташкент, B2/C1:", n_tash)
print(by_course)

## Задание 4. GROUP BY и HAVING → DataFrame

С помощью `pd.read_sql_query` получите DataFrame `city_stats` со столбцами:

- `city`, `n` — число студентов, `avg_gpa` — средний GPA (округлите до 2 знаков);
- только города, где учится **не меньше 15** студентов (`HAVING`);
- сортировка по `avg_gpa` по убыванию.

Затем получите тот же результат средствами pandas (`groupby` + фильтр) в `pandas_stats` и убедитесь, что списки городов совпадают.

<details><summary>💡 Подсказка</summary>

SQL: `SELECT city, COUNT(*) AS n, ROUND(AVG(gpa), 2) AS avg_gpa FROM students GROUP BY city HAVING COUNT(*) >= 15 ORDER BY avg_gpa DESC`. pandas: `df.groupby("city").agg(n=("gpa", "size"), avg_gpa=("gpa", "mean"))`, затем фильтр `g[g["n"] >= 15]` и `sort_values`.

</details>

In [ ]:
# Подготовительный код задания
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
pd.read_csv("data/students.csv").to_sql("students", conn, index=False)

In [ ]:
sql = """
SELECT ...
"""
city_stats = pd.read_sql_query(sql, conn)

df = pd.read_sql_query("SELECT * FROM students", conn)
pandas_stats = ...

print(city_stats)
print("Совпадает:", list(city_stats["city"]) == list(pandas_stats.index))

## Задание 5. JOIN: продажи и каталог книг

В базе (`conn`) две таблицы:

- `books` — каталог: `book, author, language, category, year, price_uzs`;
- `sales` — продажи 2026 года: `order_id, date, book, qty, price_uzs, city` (категории здесь **нет**).

Напишите запросы с соединением таблиц:

1. `by_category` (DataFrame через `pd.read_sql_query`) — для каждой категории книг: `category`, `qty` (сумма проданных экземпляров) и `revenue` (сумма `qty * price_uzs` из таблицы продаж); сортировка по `revenue` по убыванию.
2. `not_in_nukus` — список названий книг из каталога, которые **ни разу** не покупали в городе `Nukus` (`LEFT JOIN` или `NOT IN`).

<details><summary>💡 Подсказка</summary>

Выручка: `SUM(s.qty * s.price_uzs) AS revenue`, группировка `GROUP BY b.category`. Книги без продаж в Нукусе: `SELECT b.book FROM books b LEFT JOIN sales s ON s.book = b.book AND s.city = 'Nukus' WHERE s.order_id IS NULL`.

</details>

In [ ]:
# Подготовительный код задания
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
_x = pd.read_excel("data/sales.xlsx", sheet_name=None)
_x["kitoblar"].to_sql("books", conn, index=False)
_s = _x["2026"][["order_id", "date", "book", "qty", "price_uzs", "city"]].copy()
_s["date"] = _s["date"].dt.strftime("%Y-%m-%d")
_s.to_sql("sales", conn, index=False)
del _x, _s

In [ ]:
by_category = pd.read_sql_query("""
    SELECT ...
    FROM sales AS s
    JOIN books AS b ON ...
""", conn)

not_in_nukus = ...

print(by_category)
print("Не покупали в Нукусе:", not_in_nukus)

## Задание 6. UPDATE и DELETE

База `university.db` содержит таблицу `students` (200 студентов). Выполните **по порядку**:

1. Студентам с посещаемостью `attendance >= 95` увеличьте GPA на 0.1, но не выше 5.0 (в SQLite: `MIN(gpa + 0.1, 5.0)`). Число изменённых строк сохраните в `n_updated` (`cur.rowcount`).
2. Удалите студентов **4 курса** с GPA ниже 3.5 (уже после повышения). Число удалённых строк — в `n_deleted`.
3. Сохраните изменения (`commit`) и запишите в `n_left` число оставшихся студентов.

Значения 95, 5.0, 4 и 3.5 передавайте через параметры `?`.

<details><summary>💡 Подсказка</summary>

`cur.execute("UPDATE students SET gpa = MIN(gpa + 0.1, ?) WHERE attendance >= ?", (5.0, 95))`, после него `n_updated = cur.rowcount`. Удаление: `DELETE FROM students WHERE course = ? AND gpa < ?`.

</details>

In [ ]:
# Подготовительный код задания
import os, sqlite3
import pandas as pd
if os.path.exists("university.db"):
    os.remove("university.db")
_c = sqlite3.connect("university.db")
pd.read_csv("data/students.csv").to_sql("students", _c, index=False)
_c.close()
del _c

In [ ]:
import sqlite3

conn = sqlite3.connect("university.db")
cur = conn.cursor()

cur.execute("UPDATE students SET ... WHERE ...", (...))
n_updated = ...

cur.execute("DELETE FROM students WHERE ...", (...))
n_deleted = ...

n_left = ...
print("Изменено:", n_updated, "| удалено:", n_deleted, "| осталось:", n_left)
conn.close()

## Задание 7. Поток постов в SQL: to_sql и индекс

Загрузим «большие» данные в базу и ускорим поиск индексом.

1. Прочитайте `data/tweets_stream.csv` и запишите его в таблицу `posts` базы `conn = sqlite3.connect(":memory:")` методом `to_sql` (с `chunksize=500` — запись порциями).
2. Создайте индекс `idx_platform` по столбцу `platform`.
3. `plan` — результат `EXPLAIN QUERY PLAN` для запроса `SELECT COUNT(*) FROM posts WHERE platform = 'telegram'` (`fetchall()`); `uses_index` — `True`, если в плане упоминается `idx_platform`.
4. `per_platform` (DataFrame) — `platform`, `n` (число постов), `avg_likes` (средние лайки, 1 знак) по убыванию `n`.
5. `per_minute` (DataFrame) — `minute` (часы и минуты `"10:05"` — функция `substr(timestamp, 12, 5)`) и `n`, по возрастанию минуты.

<details><summary>💡 Подсказка</summary>

`tweets.to_sql("posts", conn, index=False, chunksize=500)`; `conn.execute("CREATE INDEX idx_platform ON posts(platform)")`; `uses_index = any("idx_platform" in str(row) for row in plan)`. Минута: `SELECT substr(timestamp, 12, 5) AS minute, COUNT(*) AS n FROM posts GROUP BY minute ORDER BY minute`.

</details>

In [ ]:
import sqlite3
import pandas as pd

tweets = pd.read_csv("data/tweets_stream.csv")
conn = sqlite3.connect(":memory:")
# запишите tweets в таблицу posts

# создайте индекс idx_platform

plan = ...
uses_index = ...

per_platform = ...
per_minute = ...

print(plan, uses_index)
print(per_platform)
per_minute.head()

## Самостоятельная работа

## Самостоятельная работа: «Информационная безопасность и конфиденциальность данных в технологиях Big Data»

1. Покажите на примере, как f-строка в SQL приводит к инъекции: функция `find_student(name)` с запросом `f"SELECT * FROM students WHERE full_name = '{name}'"` и ввод `' OR '1'='1`. Перепишите функцию безопасно с параметром `?`.
2. Создайте **представление** (`CREATE VIEW`) `students_public`, в котором нет ФИО и `student_id`, а только факультет, курс, город и GPA. Объясните, как представления помогают разграничивать доступ к персональным данным.
3. Сравните скорость запроса с индексом и без него на таблице из 1 000 000 строк (сгенерируйте её с помощью `numpy`), используя `time.perf_counter()`.